[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrjon/BayesFlowTutorial/blob/main/02_SIR_Diagnostics_Solution.ipynb)

# BayesFlow 2 — SIR Epidemic Models & Diagnostics — Solution

**Difficulty:** 🟡 Intermediate

> ✅ This is the **completed** notebook. Every `TODO` from the exercise is filled in. There is usually more than one good answer — yours may differ and still be correct.

In [ ]:
# ▶️  RUN THIS FIRST.  Installs this tutorial project and all its dependencies
#     (BayesFlow, JAX, Pyro, ...) from GitHub, as declared in pyproject.toml (~1-2 min).
import sys, subprocess

print("Installing the tutorial and its dependencies — this takes ~1-2 min ...")
_res = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/arrjon/BayesFlowTutorial.git"],
    capture_output=True, text=True,
)
if _res.returncode != 0:
    # A real failure — show pip's output so it can be debugged.
    print(_res.stdout); print(_res.stderr)
    raise SystemExit("Install failed — see the pip output above.")

# NOTE: pip may internally warn that google-colab pins pandas==2.2.2 while BayesFlow
# needs pandas>=2.3.3. That warning is harmless and unavoidable (the tutorial doesn't use
# google-colab's pandas features); we install quietly so it doesn't clutter the output.
print("✅ Setup complete.")
print("   If the NEXT cell raises a numpy/pandas import error, click "
      "Runtime ▸ Restart session and re-run.")

In [ ]:
import os
# BayesFlow runs on JAX, PyTorch or TensorFlow. We pick JAX here (installed on Colab).
os.environ["KERAS_BACKEND"] = "jax"
import bayesflow as bf

> 📚 **Where to read more**
> - BayesFlow website & docs: <https://bayesflow.org>
> - Example gallery (great for copy-paste starting points): <https://bayesflow.org/main/examples.html>
> - API reference: <https://bayesflow.org/main/api/bayesflow.html>

## The SIR model

We infer parameters of a stochastic epidemiological model. Individuals are **S**usceptible, **I**nfected or **R**ecovered, with transmission rate $\lambda$ and recovery rate $\mu$:

$$\frac{dS}{dt} = -\lambda \frac{S I}{N},\quad \frac{dI}{dt} = \lambda \frac{S I}{N} - \mu I,\quad \frac{dR}{dt} = \mu I.$$

We observe **daily reported new cases** for $T=14$ days. Reporting is imperfect: cases appear with a delay $L$ (an integer number of days) and are noisy, modelled with a **Negative-Binomial** observation model with dispersion $\psi$. The initial number of infected $I_0$ is also unknown. So we infer five parameters: $(\lambda, \mu, L, I_0, \psi)$.

Background reading:
- OutbreakFlow (Radev et al., 2021): <https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1009472>
- Prior/likelihood choices follow Dehning et al., *Science* 2020: <https://www.science.org/doi/10.1126/science.abb9789>

This builds directly on the linear-regression starter — the pipeline (simulator → adapter → networks → workflow → diagnostics) is the same, only the model is richer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

RNG = np.random.default_rng(2026)

## 1. Prior & simulator

The priors below come from domain knowledge (see the *Science* paper above).

In [ ]:
def prior():
    """A random draw from the joint prior over the five parameters."""
    lambd = RNG.lognormal(mean=np.log(0.4), sigma=0.5)
    mu    = RNG.lognormal(mean=np.log(1 / 8), sigma=0.2)
    D     = RNG.lognormal(mean=np.log(8), sigma=0.2)   # reporting delay L
    I0    = RNG.gamma(shape=2, scale=20)
    psi   = RNG.exponential(5)
    return {"lambd": lambd, "mu": mu, "D": D, "I0": I0, "psi": psi}

### The observation model

We split the simulator into two readable pieces:
1. `sir_mean_cases` — integrates the ODE and returns the **expected** reported new cases per day.
2. `stationary_SIR` — adds Negative-Binomial reporting noise on top.

The helper `convert_params` converts the (mean, dispersion) parameterisation of the Negative-Binomial into the `(n, p)` form NumPy expects.

In [ ]:
def convert_params(mu, phi):
    """(mean, dispersion) -> (n, p) for numpy's negative_binomial."""
    r = phi
    var = mu + 1 / r * mu ** 2
    p = (var - mu) / var
    return r, 1 - p


def sir_mean_cases(lambd, mu, D, I0, N=83e6, T=14):
    """Integrate the SIR ODE (dt=1 day) and return expected reported new cases (length T)."""
    I0 = np.ceil(I0)
    D = max(int(round(D)), 0)  # reporting delay is a non-negative integer
    S, I, R, C = [N - I0], [I0], [0], [I0]
    for t in range(1, T + D):
        I_new = lambd * (I[-1] * S[-1] / N)
        S.append(S[-1] - I_new)
        I.append(np.clip(I[-1] + I_new - mu * I[-1], 0.0, N))
        R.append(np.clip(R[-1] + mu * I[-1], 0.0, N))
        C.append(I_new)
    # cases reported from day D onwards (the reporting delay)
    return np.clip(np.array(C[D:]), 0, N)


def stationary_SIR(lambd, mu, D, I0, psi, N=83e6, T=14, eps=1e-5):
    """Forward simulation with Negative-Binomial reporting noise."""
    psi = max(float(psi), eps)  # dispersion must be positive (guards posterior re-simulation)
    mean_cases = sir_mean_cases(lambd, mu, D, I0, N=N, T=T)
    r, p = convert_params(mean_cases + eps, psi)
    return dict(cases=RNG.negative_binomial(r, p))

Combine prior and simulator, and sanity-check the output shapes.

In [ ]:
simulator = bf.make_simulator([prior, stationary_SIR])

test_sims = simulator.sample(batch_size=3)
print("lambd:", test_sims["lambd"].shape, " cases:", test_sims["cases"].shape)  # (3,1) (3,14)

### Prior predictive check

A principled Bayesian workflow always inspects the prior. Let's look at the joint prior of $\lambda$, $\mu$, $L$.

In [ ]:
prior_samples = simulator.simulators[0].sample(1000)
grid = bf.diagnostics.plots.pairs_samples(prior_samples, variable_keys=["lambd", "mu", "D"])

## 2. Adapter

The raw data are large integer counts and the parameters live on very different scales — not network-friendly. We:
- mark `cases` as a **time series** (`as_time_series`), giving it shape `(batch, T, 1)`,
- concatenate the five parameters into `inference_variables`,
- rename `cases` to `summary_variables` (they go through the summary network),
- `log1p`-transform everything (all quantities are non-negative), which also removes the need for extra standardisation.

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .as_time_series("cases")
    .concatenate(["lambd", "mu", "D", "I0", "psi"], into="inference_variables")
    .rename("cases", "summary_variables")
    .log(["inference_variables", "summary_variables"], p1=True)
)

adapted = adapter(simulator.sample(2))
print(adapted["summary_variables"].shape, adapted["inference_variables"].shape)  # (2,14,1) (2,5)

## 3. Networks & workflow

Because our data is a **sequence**, we use a small custom **GRU** summary network. Any Keras model can plug into BayesFlow by subclassing `SummaryNetwork`. The `@serializable` decorator lets us save/load the full approximator later.

For the inference network we again use a `CouplingFlow`.

In [ ]:
@bf.utils.serialization.serializable("custom")
class GRU(bf.networks.SummaryNetwork):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gru = keras.layers.GRU(64)
        self.summary_stats = keras.layers.Dense(8)

    def call(self, time_series, **kwargs):
        """Compress (batch, T, 1) time series into (batch, 8) summaries."""
        summary = self.gru(time_series, **kwargs)
        summary = self.summary_stats(summary)
        return summary


summary_net = GRU()
inference_net = bf.networks.CouplingFlow(depth=2, transform="spline")

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_net,
    summary_network=summary_net,
    standardize=None,  # not needed thanks to the log transform
)

## 4. Training (offline)

The simulator is fast, but to illustrate a realistic *low simulation budget* we pre-generate a fixed dataset and train **offline**. ~1 minute on Colab CPU.

In [ ]:
training_data = workflow.simulate(6000)
validation_data = workflow.simulate(300)

history = workflow.fit_offline(
    data=training_data, epochs=100, batch_size=64, validation_data=validation_data
)
f = bf.diagnostics.plots.loss(history)

## 5. Diagnostics — the heart of this notebook

A converged loss is **not** proof of correct inference. We validate *in silico* (on fresh simulations where we know the truth) before ever touching real data.

First, draw posterior samples for 300 fresh test datasets.

In [ ]:
num_datasets, num_samples = 300, 1000
test_sims = workflow.simulate(num_datasets)
samples = workflow.sample(conditions=test_sims, num_samples=num_samples, batch_size=50)

### 5a. Simulation-based calibration (SBC)

If the posterior is well calibrated, the rank of each true value among the posterior draws is **uniform**. Rank **histograms** should look flat; the **ECDF-difference** should stay inside the band. (ECDF is usually easier to read for many parameters.)

> SBC papers: rank histograms <https://arxiv.org/abs/1804.06788> · ECDF <https://arxiv.org/abs/2103.10522>

In [ ]:
f = bf.diagnostics.plots.calibration_histogram(samples, test_sims)
f = bf.diagnostics.plots.calibration_ecdf(samples, test_sims, difference=True)

### 5b. Recovery & the non-identifiability lesson

Now compare posterior means to ground truth.

In [ ]:
f = bf.diagnostics.plots.recovery(samples, test_sims)

> 🔎 **Interpret before you scroll.** Look at the recovery plots for $\mu$ and $L$ (`D`). They probably look *flat* — the estimate barely moves with the truth. **Is training broken?**
>
> No. From only 14 days of case counts, $\mu$ (recovery rate) and $L$ (reporting delay) are **weakly identified**: many parameter combinations produce near-identical data. No inference method — BayesFlow, MCMC, anything — can recover what the data doesn't contain. A flat recovery plot combined with **good calibration** is the signature of non-identifiability, not a bug. This is exactly why we run diagnostics.

### 5c. Automatic numerical diagnostics

The workflow can compute the 'big three' error metrics — NRMSE, calibration error, posterior contraction — in one call. **Posterior contraction near 0** for a parameter is another fingerprint of non-identifiability (the posterior is as wide as the prior).

In [ ]:
metrics = workflow.compute_default_diagnostics(test_data=test_sims, samples=samples)
metrics

In [ ]:
# The full graphical diagnostic panel in one call
figures = workflow.plot_default_diagnostics(
    test_data=test_sims, samples=samples,
    loss_kwargs={"figsize": (15, 3)}, recovery_kwargs={"figsize": (15, 3)},
    calibration_ecdf_kwargs={"figsize": (15, 3)}, z_score_contraction_kwargs={"figsize": (15, 3)},
)

## 6. Confronting 'real' data: a second model with weekend delays 📉

Real surveillance data has a notorious artefact: **fewer cases are reported on weekends**, and the backlog is dumped onto the following Monday. This produces a characteristic weekly sawtooth that our clean `stationary_SIR` simulator never generates.

Instead of downloading messy real data, we **generate our own 'real' data** from a *second, more realistic simulator* that adds this weekend effect. This lets us study a crucial question:

> **What do diagnostics look like when the model you trained on is *misspecified* for the data you apply it to?**

The redistribution rule: on a weekend day, only a fraction of that day's cases are reported; the rest are carried over to the next reporting day.

In [ ]:
def apply_weekend_effect(mean_cases, start_weekday=0, weekend_report_frac=0.4):
    """Under-report weekend cases and carry the backlog to the next day.

    start_weekday: 0=Mon, ..., 5=Sat, 6=Sun for day 0 of the series.
    weekend_report_frac: fraction of a weekend day's cases actually reported that day.
    """
    mean_cases = np.asarray(mean_cases, dtype=float)
    reported = np.zeros_like(mean_cases)
    carry = 0.0
    for t in range(len(mean_cases)):
        todays = mean_cases[t] + carry
        weekday = (start_weekday + t) % 7
        if weekday >= 5:  # Saturday or Sunday: under-report
            reported[t] = weekend_report_frac * todays
            carry = todays - reported[t]
        else:
            reported[t] = todays
            carry = 0.0
    reported[-1] += carry  # dump any leftover backlog on the last day
    return reported


def stationary_SIR_weekend(lambd, mu, D, I0, psi, N=83e6, T=14, eps=1e-5,
                           start_weekday=0, weekend_report_frac=0.4):
    """Same SIR dynamics, but with realistic weekend reporting delays."""
    psi = max(float(psi), eps)  # dispersion must be positive
    mean_cases = sir_mean_cases(lambd, mu, D, I0, N=N, T=T)
    mean_cases = apply_weekend_effect(mean_cases, start_weekday, weekend_report_frac)
    r, p = convert_params(mean_cases + eps, psi)
    return dict(cases=RNG.negative_binomial(r, p))

### Generate the 'real' observation

We pick one plausible ground-truth parameter set and simulate an observed 14-day series **from the weekend model**. This is our stand-in for real-world data. Notice the weekend dips.

In [ ]:
true_params = dict(lambd=0.45, mu=1/8, D=8, I0=30, psi=0.15)
obs = stationary_SIR_weekend(**true_params, start_weekday=0, weekend_report_frac=0.4)
obs_cases = obs["cases"]

days = np.arange(len(obs_cases))
is_weekend = ((days % 7) >= 5)
plt.figure(figsize=(9, 3))
plt.plot(days, obs_cases, "-o", color="black", label="observed (weekend model)")
plt.scatter(days[is_weekend], obs_cases[is_weekend], color="crimson", zorder=3, label="weekend")
plt.xlabel("day"); plt.ylabel("reported cases"); plt.legend(); plt.title("Our 'real' data");

### 6a. Infer with the (misspecified) plain-SIR network + posterior retrodictive check

We feed the weekend data to the network trained on the **clean** model. Then we perform a **posterior retrodictive check**: draw parameters from the posterior, re-simulate, and overlay the re-simulations on the observed data. Because the training model has no weekend effect, the re-simulations will be **smooth** and can't reproduce the sawtooth — a visible, systematic misfit.

In [ ]:
def plot_ppc(param_df, obs_cases, sim_fn, logscale=True, color="#132a70", title=""):
    """Overlay posterior re-simulations (from sim_fn) on the observed data."""
    T = len(obs_cases)
    sims = np.array([sim_fn(*row)["cases"] for row in param_df.values])
    q50 = np.quantile(sims, [0.25, 0.75], axis=0)
    q90 = np.quantile(sims, [0.05, 0.95], axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(np.median(sims, axis=0), color=color, label="median re-simulation")
    ax.fill_between(range(T), q50[0], q50[1], color=color, alpha=0.4, label="50% CI")
    ax.fill_between(range(T), q90[0], q90[1], color=color, alpha=0.2, label="90% CI")
    ax.plot(obs_cases, "o--", color="black", alpha=0.8, label="observed")
    if logscale: ax.set_yscale("log")
    ax.set_xlabel("day"); ax.set_ylabel("cases"); ax.set_title(title); ax.legend()
    return fig

In [ ]:
post = workflow.sample(conditions={"cases": obs_cases[None, :]}, num_samples=1000)
post_df = workflow.samples_to_data_frame(post)

# Re-simulate with the CLEAN model (the one the network was trained on)
f = plot_ppc(post_df, obs_cases, sim_fn=stationary_SIR,
             title="Retrodictive check — trained on clean SIR (misspecified)")

> 🔎 **What to look for.** The observed weekend points (every 6th–7th day) fall *outside* or at the edge of the prediction band while weekdays sit inside it — a regular, structured pattern of misfit. Structure in retrodictive residuals = **model misspecification**. The parameter posteriors may still look confident, which is the danger: a wrong model can be confidently wrong.

### 6b. Fix it — train on the matching model

The clean fix is to make the **simulator match reality**: retrain the approximator on the weekend model, so the network learns to expect the sawtooth. (In practice you'd also amortize over `weekend_report_frac` and `start_weekday`; here we keep them fixed for clarity.)

In [ ]:
simulator_wknd = bf.make_simulator([prior, stationary_SIR_weekend])

workflow_wknd = bf.BasicWorkflow(
    simulator=simulator_wknd, adapter=adapter,
    inference_network=bf.networks.CouplingFlow(depth=2, transform="spline"),
    summary_network=GRU(), standardize=None,
)
hist_wknd = workflow_wknd.fit_offline(
    data=workflow_wknd.simulate(6000), epochs=100, batch_size=64,
    validation_data=workflow_wknd.simulate(300), verbose=0,
)

post_w = workflow_wknd.sample(conditions={"cases": obs_cases[None, :]}, num_samples=1000)
post_w_df = workflow_wknd.samples_to_data_frame(post_w)
f = plot_ppc(post_w_df, obs_cases, sim_fn=stationary_SIR_weekend,
             title="Retrodictive check — trained on weekend model (matched)")

### 6c. (Optional) the actual COVID-19 data

For comparison, here is a loader for the *real* reported cases in Germany, early March 2020 (needs internet). The real series shows exactly the weekend dips our second model reproduces. Note that March 1, 2020 was a **Sunday**, so `start_weekday=6` when re-simulating.

In [ ]:
import datetime, pandas as pd

def load_data():
    url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
           "csse_covid_19_data/csse_covid_19_time_series/"
           "time_series_covid19_confirmed_global.csv")
    df = pd.read_csv(url)
    fmt = lambda d: f"{d.month}/{d.day}/{str(d.year)[2:4]}"
    b, e = datetime.date(2020, 3, 1), datetime.date(2020, 3, 15)
    cum = np.array(df.loc[df["Country/Region"] == "Germany", fmt(b):fmt(e)])[0]
    return np.diff(cum)

try:
    real_cases = load_data()
    post_r = workflow_wknd.sample(conditions={"cases": real_cases[None, :]}, num_samples=1000)
    post_r_df = workflow_wknd.samples_to_data_frame(post_r)
    from functools import partial
    sim_sun = partial(stationary_SIR_weekend, start_weekday=6)
    f = plot_ppc(post_r_df, real_cases, sim_fn=lambda *p: sim_sun(*p),
                 title="Real German COVID-19 cases (weekend model)")
except Exception as err:
    print("Could not download real data (offline?):", err)

## Summary

The weekend experiment is the key takeaway: **good in-silico diagnostics guarantee correct inference only for data that looks like your simulator**. Posterior retrodictive checks are your first line of defence against silent model misspecification on real data.